# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOmerSiddiqui/myInternship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is **Refresh / Content Opportunity Scoring**.

This is primarily a **ranking / scoring** problem:  
“Which pages should an editor review first?”

I can also treat it as a **binary classification** problem underneath (will this page be declining or not?) and then turn the model probability into a ranked priority score.

So the ML task type is: **ranking built on top of classification**.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / proxy I will use right now (starter data):**

`is_declining = 1` if `trend_direction == "down"`, else `0`.

This is a **proxy label**, not a perfect future outcome.  
It is calculated from the current 90-day window (`trend_pct` / `trend_direction`).

Later (with the warehouse) a stronger target would be:  
“Did this page keep declining or recover in the *next* 30 days?” using only features from the *previous* 90 days.

For this week we stay with the starter proxy because it is already in the data and matches the official starter pipeline.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary success metric: Precision@50**

Meaning: Of the 50 pages my system ranks highest for review, how many actually have the declining label?

Why this metric?
- Editors only have time to look at a small number of pages.
- We care most about the quality of the *top* of the list, not about every page.

Secondary metrics I will also report:
- ROC-AUC (overall ranking quality)
- Average Precision

A good result on the starter data would be beating the hand-written baseline (Precision@50 ≈ 0.24) by a clear margin, as the random forest already does (~0.74).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis = one row = one content page (content_id)**

The starter CSV already has this grain: 30,000 rows, each representing one anonymized page with its trailing-90-day metrics.

Below I load the data, create the proxy target column, and show a small clean dataframe so the unit of analysis is visible.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "myInternship"
    if not os.path.isdir(REPO):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/MuhammadOmerSiddiqui/myInternship.git", REPO],
            check=True
        )
    os.chdir(REPO)

csv_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(csv_path), f"CSV not found at {csv_path}. Current dir: {os.getcwd()}"

df = pd.read_csv(csv_path)

# Create the proxy target
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Show the unit of analysis clearly
print("Unit of analysis: one row = one content page (content_id)")
print("Total rows:", len(df))
print("Target distribution:")
print(df["is_declining"].value_counts(normalize=True).round(3))

# Small clean preview of the key columns
preview_cols = [
    "content_id", "client_id",
    "impressions_90d", "avg_position", "ctr",
    "content_age_days", "trend_direction", "is_declining"
]
display(df[preview_cols].head(8))

Unit of analysis: one row = one content page (content_id)
Total rows: 30000
Target distribution:
is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,trend_direction,is_declining
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,0.03,147,down,1
6,content_9a34b442b552,client_8722616204,20,7.0,0.00,90,down,1
7,content_a63219c6e95a,client_19581e27de,1724,21.2,0.06,445,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple fixed rule (for example “if age > 180 days and impressions > 500 then review”) is useful as a **baseline**, but it is not enough for three reasons:

1. **Many signals interact.** Age, trend, position, CTR, engagement, word count, and impressions all matter together. Writing one perfect if-statement for all combinations is hard.
2. **Different clients and page types behave differently.** A rule that works well for one client can be weak for another.
3. **The ranking needs continuous priority, not just yes/no.** Editors need a sorted list (page A is more urgent than page B). A model probability gives a natural ranking score; a fixed rule usually only gives coarse buckets.

The starter pipeline already shows this: the hand-written baseline reaches Precision@50 ≈ 0.24 while a simple random forest reaches ≈ 0.74. That gap is why ML is worth using here — as decision-support, not as automatic decisions.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.